# 10_silver_to_gold
Refreshes the Gold star schema from Silver for one load date.
It reads the Silver rows that were new or changed on that date (`last_load_date = date`) and MERGEs them into `dim_publisher`, `dim_game` and `fact_game_daily`, in that order so every foreign key exists before the fact row arrives.
Each target table writes its own row to `pipeline_execution_logs` (layer `Silver-to-Gold`).
Parameters: date (YYYY-MM-DD).

In [ ]:
dbutils.widgets.text("date", "")

In [ ]:
import uuid
from datetime import datetime, date as _date
from pyspark.sql import functions as F
from pyspark.sql.types import StructType, StructField, StringType, LongType, TimestampType

CATALOG, SCHEMA = "workspace", "steam"
LAYER = "Silver-to-Gold"
spark.sql(f"USE {CATALOG}.{SCHEMA}")

run_date = dbutils.widgets.get("date").strip() or str(_date.today())
datetime.strptime(run_date, "%Y-%m-%d")  # fails fast on a bad date
print(run_date)

In [ ]:
def read_silver_day():
    """Silver rows that were inserted or changed on run_date."""
    return spark.table("silver_steam").where(F.col("last_load_date") == F.to_date(F.lit(run_date)))

def pub_key():
    """Deterministic 64-bit hash of the publisher name, 0 when the publisher is missing.
    Same name always gives the same key, so re-runs and backfills never create duplicates."""
    return F.when(F.col("publisher").isNotNull(),
                  F.xxhash64(F.lower(F.trim(F.col("publisher"))))).otherwise(F.lit(0).cast("long"))

print(read_silver_day().count(), "silver rows changed on", run_date)

In [ ]:
def build_dim_publisher(src):
    return (src.where(F.col("publisher").isNotNull())
               .select(pub_key().alias("publisher_key"), F.col("publisher").alias("publisher_name"))
               .groupBy("publisher_key").agg(F.min(F.trim("publisher_name")).alias("publisher_name")))

def build_dim_game(src):
    return src.select(F.col("appid").alias("game_key"), "name", "developer",
                      F.col("first_load_date").alias("first_seen_date"),
                      F.col("last_load_date").alias("last_changed_date"))

def build_fact(src):
    price = F.col("price_usd")
    tier = (F.when(price.isNull(), 0).when(price == 0, 1)
             .when(price < 5, 2).when(price <= 20, 3).otherwise(4))
    return src.select(
        F.col("appid").alias("game_key"),
        F.date_format("last_load_date", "yyyyMMdd").cast("int").alias("date_key"),
        pub_key().alias("publisher_key"),
        tier.cast("int").alias("price_tier_key"),
        "ccu", "positive", "negative", "total_reviews", "approval_rate",
        "owners_min", "owners_max",
        ((F.col("owners_min") + F.col("owners_max")) / 2).alias("owners_mid"),
        "price_usd", "initialprice_usd", "discount_pct", "row_hash",
        F.current_timestamp().alias("load_timestamp"))

In [ ]:
def merge_dim_publisher(df):
    df.createOrReplaceTempView("pub_updates")
    r = spark.sql("""
        MERGE INTO dim_publisher t
        USING pub_updates s ON t.publisher_key = s.publisher_key
        WHEN NOT MATCHED THEN INSERT (publisher_key, publisher_name, load_timestamp)
        VALUES (s.publisher_key, s.publisher_name, current_timestamp())
    """).first()
    return r["num_inserted_rows"], r["num_updated_rows"]

def merge_dim_game(df):
    """Type 1 dimension: a changed name or developer overwrites the old value."""
    df.createOrReplaceTempView("game_updates")
    r = spark.sql("""
        MERGE INTO dim_game t
        USING game_updates s ON t.game_key = s.game_key
        WHEN MATCHED AND (NOT (t.name <=> s.name) OR NOT (t.developer <=> s.developer))
                     AND s.last_changed_date >= t.last_changed_date THEN UPDATE SET
            t.name = s.name, t.developer = s.developer,
            t.last_changed_date = s.last_changed_date, t.load_timestamp = current_timestamp()
        WHEN NOT MATCHED THEN INSERT (game_key, name, developer, first_seen_date, last_changed_date, load_timestamp)
        VALUES (s.game_key, s.name, s.developer, s.first_seen_date, s.last_changed_date, current_timestamp())
    """).first()
    return r["num_inserted_rows"], r["num_updated_rows"]

def merge_fact(df):
    """One row per (game_key, date_key). Re-running a date inserts nothing and updates only changed hashes."""
    df.createOrReplaceTempView("fact_updates")
    r = spark.sql("""
        MERGE INTO fact_game_daily t
        USING fact_updates s ON t.game_key = s.game_key AND t.date_key = s.date_key
        WHEN MATCHED AND t.row_hash <> s.row_hash THEN UPDATE SET *
        WHEN NOT MATCHED THEN INSERT *
    """).first()
    return r["num_inserted_rows"], r["num_updated_rows"]

In [ ]:
LOG_SCHEMA = StructType([
    StructField("run_id", StringType()), StructField("layer", StringType()),
    StructField("load_type", StringType()), StructField("file_processed", StringType()),
    StructField("batch_id", StringType()), StructField("start_time", TimestampType()),
    StructField("end_time", TimestampType()), StructField("status", StringType()),
    StructField("rows_inserted", LongType()), StructField("rows_updated", LongType()),
    StructField("rows_skipped_unchanged", LongType()), StructField("rows_quarantined", LongType()),
    StructField("error_message", StringType())])

def write_log(start, status, table, load_type=None, inserted=None, updated=None, skipped=None, error=None, read=None):
    """One log row per Gold table. read = Silver rows aggregated into the table;
    rows_skipped_unchanged = read minus rows written. Duration = end_time - start_time."""
    row = (str(uuid.uuid4()), LAYER, load_type, f"{table} date={run_date}" + (f" read={read}" if read is not None else ""), None, start, datetime.now(),
           status, inserted, updated, skipped, None, error)
    spark.createDataFrame([row], LOG_SCHEMA).write.mode("append").saveAsTable("pipeline_execution_logs")

In [ ]:
types = [r.load_type for r in (spark.table("bronze_steam")
         .where(F.col("load_date") == F.to_date(F.lit(run_date)))
         .select("load_type").distinct().collect())]
load_type = ",".join(sorted(t for t in types if t)) or None

# order matters: dimensions first, fact last
STEPS = [("dim_publisher", build_dim_publisher, merge_dim_publisher),
         ("dim_game", build_dim_game, merge_dim_game),
         ("fact_game_daily", build_fact, merge_fact)]

for table, build, merge in STEPS:
    start = datetime.now()
    try:
        df = build(read_silver_day())
        candidates = df.count()
        inserted, updated = merge(df)
        skipped = candidates - inserted - updated
        write_log(start, "Success", table, load_type, inserted, updated, skipped, read=candidates)
        print(f"{table:16} | read: {candidates} | inserted: {inserted} | updated: {updated} | skipped: {skipped}")
    except Exception as e:
        write_log(start, "Failure", table, load_type, error=str(e)[:2000])
        raise

In [ ]:
display(spark.sql("""
    SELECT layer, load_type, file_processed, status, rows_inserted, rows_updated,
           rows_skipped_unchanged, start_time, end_time
    FROM pipeline_execution_logs WHERE layer = 'Silver-to-Gold'
    ORDER BY start_time DESC LIMIT 6
"""))
display(spark.sql("""
    SELECT (SELECT COUNT(*) FROM fact_game_daily) AS fact_rows,
           (SELECT COUNT(*) FROM dim_game)        AS games,
           (SELECT COUNT(*) FROM dim_publisher)   AS publishers
"""))